# Energy, Density, and Score

Chapter 17: Score Function and Energy-Based Models · *Mathematics for AI and Machine Learning* by Fuheng Wu

Linked triple view: double-well energy, p∝e^{−E}, and score field.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-17/three-views/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's double well: E(x) = x^4/4 - x^2, p(x) ∝ exp(-E(x)), score s(x) = -E'(x)


def energy(x):
    return 0.25 * x ** 4 - x ** 2


# ── Score = minus the energy gradient (autograd) = 2x - x^3 ─────────────────
x = torch.linspace(-2.5, 2.5, 11, dtype=torch.float64, requires_grad=True)
(dE,) = torch.autograd.grad(energy(x).sum(), x)
print("max |(-dE/dx) - (2x - x^3)| =", (-dE - (2 * x.detach() - x.detach() ** 3)).abs().max().item())

# The score vanishes where the energy is flat: two modes and one saddle
for r, label in ((-math.sqrt(2), "mode"), (0.0, "unstable point"), (math.sqrt(2), "mode")):
    s = 2 * r - r ** 3
    curvature = 3 * r ** 2 - 2          # E''(r)
    print(f"x = {r:+.4f}: score {s:+.1e}, E'' = {curvature:+.1f} -> {label}")

# ── Normalize the density numerically ───────────────────────────────────────
grid = torch.linspace(-4, 4, 40001, dtype=torch.float64)
unnorm = torch.exp(-energy(grid))
Z = torch.trapezoid(unnorm, grid)
p = unnorm / Z
print(f"\nZ = {Z.item():.6f}")
print("density at the modes vs at 0:", (torch.exp(-energy(torch.tensor(math.sqrt(2)))) / Z).item(),
      (1 / Z).item(), " ratio e^1 =", math.e)
print("mass in x > 0:", torch.trapezoid(p[grid > 0], grid[grid > 0]).item(), "(symmetric wells)")

max |(-dE/dx) - (2x - x^3)| = 0.0
x = -1.4142: score +4.4e-16, E'' = +4.0 -> mode
x = +0.0000: score +0.0e+00, E'' = -2.0 -> unstable point
x = +1.4142: score -4.4e-16, E'' = +4.0 -> mode

Z = 7.587482
density at the modes vs at 0: 0.35825872303952083 0.13179602282893657  ratio e^1 = 2.718281828459045
mass in x > 0: 0.49997364079490714 (symmetric wells)
